# שיעור 32 — פונקציות ומימוש עצמאי של KNN

**הקשר ללמידת מכונה:** דודו יודע להפעיל ספרייה. כעת יוכל להסביר ולבדוק כיצד מרחק,מיון והצבעה הופכים לתחזית,בלי להסתיר אותם בקריאת predict.

**עוגן:** [סיכום KNN מפגש 5](../../07-KNN/Meeting-05-Didactic-Summary.md),שיעור 23 ו[מדריך המימוש ב־DL](../../../03-Deep-Learning-5-Units/07-Classification-KNN/Meeting-05-KNN-Implementation-Guide.md). זו גרסה בלולאות לפני Broadcasting,לא העתק מחברת המרצה.

המחברת היא התאמה דידקטית חדשה. אין קובץ מקור חדש מהמרצה ולכן אין Exact-Original מלאכותי. אין לייחס לה אישור כיסוי מחוון עדכני.

## למה לומדים את זה עכשיו?
נמליץ על מסלול פעילות מתוך שכנים דומים; הפונקציה מפרקת כיצד מתקבלת ההחלטה.
**בתהליך הפרויקט:** הבנת האלגוריתם.
**מסגרת פעילה:** 2 שעות לימוד — 90 דקות, כולל דוגמה ותרגול עצמאי. המורה בוחר את משימות הליבה שבמדריך; שאר הפעילויות לחזרה או לעיון, ללא תוספת שעות.

## מושגים וידע קודם
פונקציה מגדירה פעולה לשימוש חוזר; פרמטר הוא קלט,return הוא פלט ולא print. train ויעד אינם חלק מאותו וקטור מדידה.  מימוש פשוט סורק את כל דוגמאות האימון לכל שאילתה; ספרייה עשויה לייעל חיפוש.

## הסבר מתמטי
d(u,v)=√Σ(u_j−v_j)². בוחרים K מרחקים קטנים ומצביעים לפי תוויותיהם. אצלנו מיון מרחקים יציב ובתיקו קולות בוחרים את התווית הראשונה בסדר המיון של mode; זו מדיניות מפורשת ולא אמת מתמטית יחידה. בתיקו מרחק בגבול K תוצאת ספרייה עשויה להשתנות לפי סדר נתונים. לא משווים תוצאות בתנאים שונים.

## נתונים ופונקציית מרחק
הנתונים מומצאים,באותו סולם וללא השלמה/תקנון; אין כאן מבחן Generalization.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.neighbors import KNeighborsClassifier
X_train=np.array([[3.,2.],[5.,3.],[5.,5.],[7.,3.],[3.,8.]])
y_train=np.array(["Red","Blue","Blue","Red","Red"])
queries=np.array([[3.,3.]])
def euclidean_distance(u,v):
    total=0.0
    for j in range(len(u)):
        total=total+(float(u[j])-float(v[j]))**2
    return total**0.5
print("Distance check:",euclidean_distance([1,2],[4,6]))

## סיווג יחיד
שומרים לכל מרחק תווית מתאימה. K חייב להיות שלם בין 1 למספר דוגמאות האימון; כאן הערכים נתונים ותקינים.

In [ ]:
def predict_one(train_X,train_y,query,k):
    distances=[]
    for row in train_X:
        distances.append(euclidean_distance(row,query))
    neighbors=pd.DataFrame({"distance":distances,"label":train_y})
    nearest=neighbors.sort_values("distance",kind="stable").head(k)
    return nearest["label"].mode().iloc[0]
manual_predictions=[]
for q in queries:
    manual_predictions.append(predict_one(X_train,y_train,q,3))
library=KNeighborsClassifier(n_neighbors=3,weights="uniform",metric="euclidean")
library.fit(X_train,y_train)
print("Manual:",manual_predictions,"Library:",library.predict(queries))

## מסקנה ובדיקת הבנה
תעדו מה נכנס,מה חושב,מה התקבל ומה אי אפשר להסיק. כתבו שני משפטים על הקשר לפרויקט.
לא שוכחים return; לא ממיינים מרחקים בלי התוויות; לא מצביעים בכל השורות במקום K; לא מייחסים Accuracy לשאילתה שאין לה אמת. הפונקציות אינן estimator של sklearn ואינן מוכנסות ל־GridSearchCV. נדרש קלט דו־ממדי,אימון לא ריק וממדים תואמים; אין טיפול כללי בשגיאות בתרגיל.

**כרטיס יציאה:** הסבירו פעולה אחת ללא קריאת הקוד ומגבלה אחת של הדוגמה. דוגמאות מומצאות אינן ראיה על אוכלוסייה אמיתית.

## כמה שאילתות והשוואת ייצוג
כל קריאה מחזירה תווית אחת. החיסור הווקטורי הבא מחשב אותם מרחקים כמו הלולאה, ואינו מודל נוסף.

In [ ]:
two_queries=np.array([[3.,3.],[6.,4.]])
for query in two_queries:
    print("Query:", query, "Prediction:", predict_one(X_train,y_train,query,3))
loop_distances=[]
for row in X_train:
    loop_distances.append(euclidean_distance(row,queries[0]))
vector_distances=np.sqrt(((X_train-queries[0])**2).sum(axis=1))
print("Same distances:",np.allclose(loop_distances,vector_distances))